# ML

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings("ignore")
 
from sklearn.pipeline          import Pipeline
from sklearn.compose           import ColumnTransformer
from sklearn.preprocessing     import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.linear_model      import LogisticRegression
from sklearn.ensemble          import RandomForestClassifier
from sklearn.model_selection   import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics           import (accuracy_score, f1_score, roc_auc_score, auc , 
                                       confusion_matrix, classification_report,
                                       roc_curve, ConfusionMatrixDisplay)
from imblearn.over_sampling    import SMOTE
import xgboost as xgb
from imblearn.pipeline         import Pipeline as ImbPipeline

In [9]:
df = pd.read_csv('clean_data.csv')
display(df.head(5))

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Name,Customer Country,Customer Segment,...,Product Name,Product Price,shipping date (DateOrders),Shipping Mode,Order_process_time,delay,is_delay,order_month,order_day,order_hour
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,Sporting Goods,Puerto Rico,Consumer,...,Smart watch,327.75,2018-02-03 22:56:00,Standard Class,3,-1,False,1,Wednesday,22
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,Sporting Goods,Puerto Rico,Consumer,...,Smart watch,327.75,2018-01-18 12:27:00,Standard Class,5,1,True,1,Saturday,12
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,Sporting Goods,EE. UU.,Consumer,...,Smart watch,327.75,2018-01-17 12:06:00,Standard Class,4,0,False,1,Saturday,12
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,Sporting Goods,EE. UU.,Home Office,...,Smart watch,327.75,2018-01-16 11:45:00,Standard Class,3,-1,False,1,Saturday,11
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,Sporting Goods,Puerto Rico,Corporate,...,Smart watch,327.75,2018-01-15 11:24:00,Standard Class,2,-2,False,1,Saturday,11


In [5]:
print(df['Late_delivery_risk'].value_counts()
)
print(df['Late_delivery_risk'].value_counts(normalize=True))

Late_delivery_risk
1    98977
0    73788
Name: count, dtype: int64
Late_delivery_risk
1    0.5729
0    0.4271
Name: proportion, dtype: float64


In [6]:
df.columns

Index(['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)',
       'Benefit per order', 'Sales per customer', 'Delivery Status',
       'Late_delivery_risk', 'Category Name', 'Customer Country',
       'Customer Segment', 'Customer Street', 'Department Id',
       'Department Name', 'order date (DateOrders)',
       'Order Item Product Price', 'Order Item Profit Ratio', 'Sales',
       'Order Profit Per Order', 'Order Region', 'Order Status',
       'Product Name', 'Product Price', 'shipping date (DateOrders)',
       'Shipping Mode', 'Order_process_time', 'delay', 'is_delay',
       'order_month', 'order_day', 'order_hour'],
      dtype='object')

In [11]:
df.shape

(172765, 30)

In [20]:
df['is_weekend'] = df['order_day'].isin(['Saturday', 'Sunday']).astype(int)
df['is_peak_hour'] = df['order_hour'].between(9, 18).astype(int)

# Frequency Encoding (for high-cardinality)
freq_cols = ['Order Region', 'Category Name']

for col in freq_cols:
    freq_map = df[col].value_counts(normalize=True)
    df[col + "_freq"] = df[col].map(freq_map)

In [23]:
df = df.drop(columns = freq_cols)

In [25]:
df.columns

Index(['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)',
       'Benefit per order', 'Sales per customer', 'Delivery Status',
       'Late_delivery_risk', 'Customer Country', 'Customer Segment',
       'Customer Street', 'Department Id', 'Department Name',
       'order date (DateOrders)', 'Order Item Product Price',
       'Order Item Profit Ratio', 'Sales', 'Order Profit Per Order',
       'Order Status', 'Product Name', 'Product Price',
       'shipping date (DateOrders)', 'Shipping Mode', 'Order_process_time',
       'delay', 'is_delay', 'order_month', 'order_day', 'order_hour',
       'is_weekend', 'is_peak_hour', 'Order Region_freq',
       'Category Name_freq'],
      dtype='object')

In [27]:
drop_col = ['Days for shipping (real)' , 'Days for shipment (scheduled)' , 'Benefit per order' , 'Sales per customer' , 'Delivery Status' ,'Customer Country' , 'Customer Street','Department Id' , 'order date (DateOrders)', 'Order Item Product Price','Order Item Profit Ratio', 'Sales', 'Order Profit Per Order','Order Status', 'Product Name', 'Product Price','shipping date (DateOrders)','Order_process_time',
'delay', 'is_delay' , 'order_day']
df = df.drop(columns = drop_col)

In [28]:
df.columns

Index(['Type', 'Late_delivery_risk', 'Customer Segment', 'Department Name',
       'Shipping Mode', 'order_month', 'order_hour', 'is_weekend',
       'is_peak_hour', 'Order Region_freq', 'Category Name_freq'],
      dtype='object')

In [29]:
df.to_csv('ml_dataset.csv' , index=False)